# Momentum on the CAC 40

This notebook tests a momentum strategy on CAC 40 stocks. At the end of each month I rank the stocks by their return between t-12 and t-1, buy the top 20%, short the bottom 20% and hold the portfolio for one month. I want to know whether it still pays once transaction costs are paid.

In [ ]:
import sys; sys.path.insert(0, "..")
import os
import matplotlib.pyplot as plt
import pandas as pd

from src.data import CAC40_TICKERS, BENCHMARK, download_prices, to_monthly, synthetic_prices
from src.strategy import backtest, long_short_weights, momentum_signal
from src import metrics as m

plt.rcParams["figure.figsize"] = (9, 4)

## 1. Data

In [ ]:
if os.environ.get("DEMO"):
    prices = synthetic_prices()
else:
    prices = download_prices(CAC40_TICKERS + [BENCHMARK], start="2005-01-01", cache="../data/prices.csv")

index_px = prices.pop(BENCHMARK) if BENCHMARK in prices else None
monthly = to_monthly(prices)
print(f"{monthly.shape[1]} stocks, {monthly.index[0]:%Y-%m} -> {monthly.index[-1]:%Y-%m}")
monthly.iloc[:, :5].tail()

## 2. Baseline backtest

12-1 window, quintiles, 10 basis points of costs. I compare it with an equal-weighted portfolio of the same stocks.

In [ ]:
res = backtest(monthly, lookback=12, skip=1, quantile=0.2, cost_bps=10)
bench_ew = monthly.pct_change().mean(axis=1).reindex(res.index)

fig, ax = plt.subplots()
(1 + res["gross"]).cumprod().plot(ax=ax, label="Momentum L/S (gross)")
(1 + res["net"]).cumprod().plot(ax=ax, label="Momentum L/S (net of costs)")
(1 + bench_ew).cumprod().plot(ax=ax, label="Equal-weighted (long only)", alpha=.7)
ax.set_title("Cumulative performance (base 1)"); ax.legend(); plt.show()

In [ ]:
table = pd.concat([
    m.summary(res["gross"], "Momentum gross"),
    m.summary(res["net"], "Momentum net"),
    m.summary(bench_ew, "Equal-weighted"),
], axis=1)
print(f"Average monthly turnover: {res['turnover'].mean():.2f}")
table.round(3)

### Worst months

I look at which stocks weighed the most in the months where the strategy loses the most, to see whether a drop comes from a bad price.

In [ ]:
w_held = long_short_weights(momentum_signal(monthly, 12, 1), 0.2).shift(1)
contrib = w_held * monthly.pct_change()
for d, r in res["net"].nsmallest(5).items():
    worst = contrib.loc[d].nsmallest(3)
    print(f"{d:%Y-%m}: {r:+.1%} | worst contributions: " + ", ".join(f"{t} {v:+.1%}" for t, v in worst.items()))

## 3. Alpha and beta

I regress the net strategy returns on the equal-weighted portfolio returns, with Newey-West standard errors.

In [ ]:
pd.DataFrame({"vs equal-weighted": m.alpha_beta(res["net"], bench_ew)}).round(3)

## 4. Robustness

Does the result hold when the formation window or the cost level changes?

In [ ]:
rows = {}
for lb in [3, 6, 9, 12]:
    r = backtest(monthly, lookback=lb)["net"]
    rows[f"{lb}-month window"] = {"Ann. return": m.annual_return(r), "Sharpe": m.sharpe(r), "Max DD": m.max_drawdown(r)}
pd.DataFrame(rows).T.round(3)

In [ ]:
rows = {}
for c in [0, 10, 20, 50]:
    r = backtest(monthly, cost_bps=c)["net"]
    rows[f"{c} bps"] = {"Ann. return": m.annual_return(r), "Sharpe": m.sharpe(r)}
pd.DataFrame(rows).T.round(3)

## 5. Out-of-sample test

I pick the formation window on the first 60% of the months, then look at what that choice gives on the rest of the period.

In [ ]:
split = monthly.index[int(len(monthly) * 0.6)]
train_sharpe = {lb: m.sharpe(backtest(monthly, lookback=lb)["net"].loc[:split]) for lb in [3, 6, 9, 12]}
best = max(train_sharpe, key=train_sharpe.get)
test = backtest(monthly, lookback=best)["net"].loc[split:]
print(f"Split: {split:%Y-%m} | best window (train): {best} months")
print(f"Train Sharpe: {train_sharpe[best]:.2f} | Test Sharpe: {m.sharpe(test):.2f}")

## 6. Limitations

- The stocks are the current members of the CAC 40. Companies that left the index are not in the sample.
- With about forty stocks, each quintile holds around 8 names, so the results are noisy.
- Costs are proportional to turnover. The borrowing cost of short positions is ignored.
- The risk-free rate is taken as zero in the Sharpe ratio.
- I did not use the Fama-French factors, which would be a natural next step.